# _common - shared helpers

Included by every pipeline notebook with `%run ./_common` (not by `01_setup`, which must run before the marker exists). Defines:

* `CATALOG`, `JOB_RUN_ID`, `IN_JOB` (from widgets / job parameters), `PIPELINE_VOLUME` (checkpoints, config)
* `table(schema, name)` - fully qualified, validated table names
* `SETUP_VERSION`, `require_setup()` - stops with "Run the 'JobSeeker setup' job (01_setup) first" unless `01_setup` set TBLPROPERTY `jobseeker.setup_version` >= `SETUP_VERSION` on `ops.pipeline_runs`. Called once below, so every notebook that includes this one is checked; all DDL lives in `01_setup`
* `log_run(...)` - writes one row per task run to `ops.pipeline_runs`
* `task_run(task_name)` - context manager that times a task and logs SUCCESS / FAILED
* `stop_if_job()` - ends scheduled runs before the diagnostic "Inspect" cells (saves Free Edition compute)

In [ ]:
import json
import re
from contextlib import contextmanager
from datetime import datetime, timezone

dbutils.widgets.text("catalog", "jobseeker", "Catalog")
dbutils.widgets.text("job_run_id", "", "Job run id (set by the job)")

CATALOG = dbutils.widgets.get("catalog").strip()
if not re.fullmatch(r"[A-Za-z0-9_]+", CATALOG):
    raise ValueError(f"Invalid catalog name: {CATALOG!r}")

# The job passes job_run_id; interactive runs leave it empty
IN_JOB = bool(dbutils.widgets.get("job_run_id").strip())
JOB_RUN_ID = dbutils.widgets.get("job_run_id").strip() or f"manual-{datetime.now(timezone.utc):%Y%m%dT%H%M%S}"
# The run id is embedded in SQL text by several notebooks, so only allow a safe character set
if not re.fullmatch(r"[A-Za-z0-9_.:-]{1,64}", JOB_RUN_ID):
    raise ValueError(f"Invalid job_run_id: {JOB_RUN_ID!r}")
PIPELINE_VOLUME = f"/Volumes/{CATALOG}/ops/pipeline"

# Minimum 01_setup version these notebooks need (SETUP_VERSION in 01_setup); bump both together
SETUP_VERSION = 2
_setup_checked = False


def table(schema, name):
    if not re.fullmatch(r"[A-Za-z0-9_]+", schema) or not re.fullmatch(r"[A-Za-z0-9_]+", name):
        raise ValueError(f"Invalid table identifier: {schema}.{name}")
    return f"{CATALOG}.{schema}.{name}"


def require_setup():
    """Metadata-only check of the marker 01_setup writes last; the pipeline never creates or alters tables itself."""
    global _setup_checked
    if _setup_checked:
        return
    hint = f"Run the 'JobSeeker setup' job (01_setup) first: catalog {CATALOG} needs setup version {SETUP_VERSION}"
    try:
        properties = {row["key"]: row["value"] for row in spark.sql(f"SHOW TBLPROPERTIES {table('ops', 'pipeline_runs')}").collect()}
    except Exception as exc:
        raise RuntimeError(f"{hint} ({type(exc).__name__}: {str(exc).splitlines()[0][:200]})") from exc
    found = properties.get("jobseeker.setup_version", "")
    if not found.isdigit() or int(found) < SETUP_VERSION:
        raise RuntimeError(f"{hint}, found {found or 'none'}")
    _setup_checked = True


def log_run(task, status, started_at, files_read=None, rows_in=None, rows_out=None, rows_rejected=None, details=None, message=None):
    row = [(
        JOB_RUN_ID, task, status, started_at, datetime.now(timezone.utc),
        files_read, rows_in, rows_out, rows_rejected,
        json.dumps(details or {}, default=str), (message or "")[:2000],
    )]
    schema = (
        "job_run_id STRING, task STRING, status STRING, started_at TIMESTAMP, finished_at TIMESTAMP, "
        "files_read INT, rows_in BIGINT, rows_out BIGINT, rows_rejected BIGINT, details STRING, message STRING"
    )
    spark.createDataFrame(row, schema).write.mode("append").saveAsTable(table("ops", "pipeline_runs"))


@contextmanager
def task_run(task):
    """Collects metrics in a dict and logs SUCCESS or FAILED to ops.pipeline_runs."""
    metrics = {"details": {}}
    started_at = datetime.now(timezone.utc)
    try:
        yield metrics
    except Exception as exc:
        log_run(task, "FAILED", started_at, message=f"{type(exc).__name__}: {exc}", details=metrics.get("details"))
        raise
    else:
        log_run(
            task, "SUCCESS", started_at,
            files_read=metrics.get("files_read"), rows_in=metrics.get("rows_in"), rows_out=metrics.get("rows_out"),
            rows_rejected=metrics.get("rows_rejected"), details=metrics.get("details"), message=metrics.get("message"),
        )
        print(f"[{task}] SUCCESS run={JOB_RUN_ID} {({k: v for k, v in metrics.items() if k != 'details'})}")


def stop_if_job():
    """Ends a scheduled job run before the 'Inspect' cells: they only display diagnostics, and every Spark action
    they trigger counts against the Free Edition compute quota. Interactive runs continue into them."""
    if IN_JOB:
        dbutils.notebook.exit("done")


require_setup()
print(f"catalog={CATALOG} job_run_id={JOB_RUN_ID} in_job={IN_JOB} setup_version>={SETUP_VERSION}")